In [2]:
import datetime
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import rasterio
from matplotlib import pyplot as plt
from shapely.geometry import box

from estuary.util.img import broad_band, contrast_stretch

In [3]:
def s2_awei_img(bands):
    green = bands[2]
    NIR = bands[7]
    SWIR = bands[10]
    MIR = bands[11]
    AWEI = (4 * (green - MIR)) - (0.25 * NIR - 2.75 * SWIR)
    return contrast_stretch(np.log10(1 + AWEI - AWEI.min()))


def s2_broad_band(bands):
    blue = np.log10(1 + bands[10:12].mean(axis=0))
    green = np.log10(1 + bands[0:3].mean(axis=0))
    red = np.log10(1 + bands[3:9].mean(axis=0))

    img = np.dstack([red, green, blue])
    return contrast_stretch(img.transpose((2, 0, 1))).transpose((1, 2, 0))


def s2_tri_stimulus(bands):
    red_recipe = np.log10(
        1.0
        + 0.01 * bands[0]
        + 0.09 * bands[1]
        + 0.35 * bands[2]
        + 0.04 * bands[3]
        + 0.01 * bands[4]
        + 0.59 * bands[5]
        + 0.85 * bands[6]
        + 0.12 * bands[7]
        + 0.07 * bands[9]
        + 0.04 * bands[10]
    )
    green_recipe = np.log10(
        1.0
        + 0.26 * bands[2]
        + 0.21 * bands[3]
        + 0.50 * bands[4]
        + 1.00 * bands[5]
        + 0.38 * bands[6]
        + 0.04 * bands[7]
        + 0.03 * bands[9]
        + 0.02 * bands[10]
    )
    blue_recipe = np.log10(
        1.0
        + 0.07 * bands[0]
        + 0.28 * bands[1]
        + 1.77 * bands[2]
        + 0.47 * bands[3]
        + 0.16 * bands[4]
    )

    rgb_tri = np.dstack((red_recipe, green_recipe, blue_recipe))
    mn = rgb_tri.min()
    rgb_tri -= mn
    rgb_tri /= rgb_tri.max()
    return contrast_stretch(rgb_tri.transpose((2, 0, 1))).transpose((1, 2, 0))


S2_BAND_NAMES = [
    "B01",
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B09",
    "B11",
    "B12",
]


def s2_imgs(s2_dir):
    bands = []
    for name in S2_BAND_NAMES:
        file = list(s2_dir.glob(f"*_{name}_*"))[0]
        with rasterio.open(file) as src:
            bands.append(src.read(1))

    bands = np.array(bands)
    awei = s2_awei_img(bands)
    img = s2_broad_band(bands)
    tri = s2_tri_stimulus(bands)

    return img, tri, awei


def parse_file_datetime(filepath):
    date_format = "%Y%m%d_%H%M%S_%f"

    datestr = "_".join(filepath.name.split("_")[:3])
    return datetime.datetime.strptime(datestr, date_format).date()


def parse_s2_datetime(filepath):
    return datetime.datetime.strptime(filepath.name, "%Y-%m-%d-%H").date()

In [4]:
BASE = Path("/Users/kyledorman/data/estuary_viz/")
GRID_PATH = BASE / "grids/"
SENTINEL_PATH = Path("/Users/kyledorman/data/estuary_viz/sentinel")

for p in GRID_PATH.iterdir():
    print(p.stem)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kyledorman/data/estuary_viz/grids'

# los_penasquitos_lagoon

In [ ]:
filename = "los_penasquitos_lagoon"

gdf = gpd.read_file(GRID_PATH / f"{filename}.geojson")
gdf["filename"] = filename

# Compute the bounding box of all polygons
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate the center of the bounding box
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Step 5: Create a Folium Map and Overlay
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry,
        name=row.filename,
        # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
        popup=folium.Popup(row.filename, parse_html=True),
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.5,
        },
    ).add_to(m)

# Display the map (if running in a Jupyter Notebook)
m

In [ ]:
filename = "los_penasquitos_lagoon"

tifs = list((BASE / "results").glob(f"*/*/*/{filename}/files/*AnalyticMS_SR*.tif"))

s2_paths = list((SENTINEL_PATH / filename).iterdir())

all_dates = sorted(
    list(
        set(parse_s2_datetime(ip) for ip in s2_paths) | set(parse_file_datetime(ip) for ip in tifs)
    )
)

rows = len(all_dates)
cols = 2

fig, axs = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows))
for ax in axs.flatten():
    ax.axis("off")

for i, capture_date in enumerate(all_dates):
    axs[i, 0].set_title(str(capture_date))

    t_path = next((ip for ip in tifs if parse_file_datetime(ip) == capture_date), None)
    if t_path is not None:
        with rasterio.open(t_path) as src:
            data = src.read(list(range(1, 9)), masked=True)
            mask = data.mask[0]
            data = data.data
        img = broad_band(data, mask)

        axs[i, 0].imshow(img)

    s2_path = next((ip for ip in s2_paths if parse_s2_datetime(ip) == capture_date), None)
    if s2_path is not None:
        img, tri, awei = s2_imgs(s2_path)

        axs[i, 1].imshow(tri)
#         axs[i, 2].imshow(tri)
#         axs[i, 3].imshow(awei)

fig.tight_layout()

# santa_margarita

In [ ]:
filename = "santa_margarita"

gdf = gpd.read_file(GRID_PATH / f"{filename}.geojson")
gdf["filename"] = filename

# Compute the bounding box of all polygons
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate the center of the bounding box
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Step 5: Create a Folium Map and Overlay
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry,
        name=row.filename,
        # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
        popup=folium.Popup(row.filename, parse_html=True),
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.5,
        },
    ).add_to(m)

# Display the map (if running in a Jupyter Notebook)
m

In [ ]:
filename = "santa_margarita"

tifs = list((BASE / "results").glob(f"*/*/*/{filename}/files/*AnalyticMS_SR*.tif"))

s2_paths = list((SENTINEL_PATH / filename).iterdir())

all_dates = sorted(
    list(
        set(parse_s2_datetime(ip) for ip in s2_paths) | set(parse_file_datetime(ip) for ip in tifs)
    )
)

rows = len(all_dates)
cols = 2

fig, axs = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows))
for ax in axs.flatten():
    ax.axis("off")

for i, capture_date in enumerate(all_dates):
    axs[i, 0].set_title(str(capture_date))

    t_path = next((ip for ip in tifs if parse_file_datetime(ip) == capture_date), None)
    if t_path is not None:
        with rasterio.open(t_path) as src:
            data = src.read(list(range(1, 9)), masked=True)
            mask = data.mask[0]
            data = data.data
        img = broad_band(data, mask)

        axs[i, 0].imshow(img[100:500, 100:500])

    s2_path = next((ip for ip in s2_paths if parse_s2_datetime(ip) == capture_date), None)
    if s2_path is not None:
        img, tri, awei = s2_imgs(s2_path)

        start = int(100 / 2.8)
        end = start + int(400 / 2.8)

        axs[i, 1].imshow(tri[start:end, start:end])
#         axs[i, 2].imshow(tri)
#         axs[i, 3].imshow(awei)

fig.tight_layout()

# malibu_lagoon

In [ ]:
filename = "malibu_lagoon"

gdf = gpd.read_file(GRID_PATH / f"{filename}.geojson")
gdf["filename"] = filename

# Compute the bounding box of all polygons
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate the center of the bounding box
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Step 5: Create a Folium Map and Overlay
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry,
        name=row.filename,
        # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
        popup=folium.Popup(row.filename, parse_html=True),
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.5,
        },
    ).add_to(m)

# Display the map (if running in a Jupyter Notebook)
m

In [ ]:
filename = "malibu_lagoon"

tifs = list((BASE / "results").glob(f"*/*/*/{filename}/files/*AnalyticMS_SR*.tif"))

s2_paths = list((SENTINEL_PATH / filename).iterdir())

all_dates = sorted(
    list(
        set(parse_s2_datetime(ip) for ip in s2_paths) | set(parse_file_datetime(ip) for ip in tifs)
    )
)

rows = len(all_dates)
cols = 2

fig, axs = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows))
for ax in axs.flatten():
    ax.axis("off")

for i, capture_date in enumerate(all_dates):
    axs[i, 0].set_title(str(capture_date))

    t_path = next((ip for ip in tifs if parse_file_datetime(ip) == capture_date), None)
    if t_path is not None:
        with rasterio.open(t_path) as src:
            data = src.read(list(range(1, 9)), masked=True)
            mask = data.mask[0]
            data = data.data
        img = broad_band(data, mask)

        axs[i, 0].imshow(img)

    s2_path = next((ip for ip in s2_paths if parse_s2_datetime(ip) == capture_date), None)
    if s2_path is not None:
        img, tri, awei = s2_imgs(s2_path)

        axs[i, 1].imshow(tri)
#         axs[i, 2].imshow(tri)
#         axs[i, 3].imshow(awei)

fig.tight_layout()

# mugu_lagoon

In [ ]:
filename = "mugu_lagoon"

gdf = gpd.read_file(GRID_PATH / f"{filename}.geojson")
gdf["filename"] = filename

# Compute the bounding box of all polygons
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate the center of the bounding box
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Step 5: Create a Folium Map and Overlay
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry,
        name=row.filename,
        # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
        popup=folium.Popup(row.filename, parse_html=True),
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.5,
        },
    ).add_to(m)

# Display the map (if running in a Jupyter Notebook)
m

In [ ]:
filename = "mugu_lagoon"

tifs = list((BASE / "results").glob(f"*/*/*/{filename}/files/*AnalyticMS_SR*.tif"))

s2_paths = list((SENTINEL_PATH / filename).iterdir())

all_dates = sorted(
    list(
        set(parse_s2_datetime(ip) for ip in s2_paths) | set(parse_file_datetime(ip) for ip in tifs)
    )
)
all_dates = all_dates[2:]

rows = len(all_dates)
cols = 2

fig, axs = plt.subplots(rows, cols, figsize=(7 * cols, 7 * rows))
for ax in axs.flatten():
    ax.axis("off")

for i, capture_date in enumerate(all_dates):
    axs[i, 0].set_title(str(capture_date))

    t_path = next((ip for ip in tifs if parse_file_datetime(ip) == capture_date), None)
    if t_path is not None:
        with rasterio.open(t_path) as src:
            data = src.read(list(range(1, 9)), masked=True)
            mask = data.mask[0]
            data = data.data
        img = broad_band(data, mask)[500:900, 400:800]

        axs[i, 0].imshow(img)

    s2_path = next((ip for ip in s2_paths if parse_s2_datetime(ip) == capture_date), None)
    if s2_path is not None:
        img, tri, awei = s2_imgs(s2_path)

        y_start = int(500 / 2.8)
        y_end = y_start + int(400 / 2.8)
        x_start = int(400 / 2.8)
        x_end = x_start + int(400 / 2.8)

        axs[i, 1].imshow(tri[y_start:y_end, x_start:x_end])
#         axs[i, 2].imshow(tri)
#         axs[i, 3].imshow(awei)

fig.tight_layout()

In [ ]:
import geopandas as gpd
from pyproj import CRS, Transformer


def get_utm_crs(lon, lat):
    utm_zone = int((lon + 180) / 6) + 1
    is_northern = lat >= 0
    epsg_code = 32600 + utm_zone if is_northern else 32700 + utm_zone
    return CRS.from_epsg(epsg_code)


def point_to_box_gdf(lon, lat, width_m, height_m, crs_out="EPSG:4326"):
    """
    Convert a lat/lon point to a rectangular polygon of given size (meters) and return a GeoDataFrame.

    Args:
        lat (float): Latitude of center point.
        lon (float): Longitude of center point.
        width_m (float): Width of the box in meters.
        height_m (float): Height of the box in meters.
        crs_out (str): Desired output CRS (default WGS84)

    Returns:
        geopandas.GeoDataFrame: A GDF with a single polygon geometry.
    """
    # Use a local UTM zone for accurate meters
    utm_crs = get_utm_crs(lon, lat)
    transformer_to_utm = Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True)

    # Transform center point to UTM
    x_center, y_center = transformer_to_utm.transform(lon, lat)

    # Create box around the center
    half_w = width_m / 2
    half_h = height_m / 2
    x_min, x_max = x_center - half_w, x_center + half_w
    y_min, y_max = y_center - half_h, y_center + half_h

    # Create box in UTM
    poly_utm = box(x_min, y_min, x_max, y_max)

    # Put it into a GeoDataFrame
    gdf_utm = gpd.GeoDataFrame(geometry=[poly_utm], crs=utm_crs)

    # Reproject the entire geometry properly
    gdf_latlon = gdf_utm.to_crs("EPSG:4326")

    return gdf_latlon

In [ ]:
filename = "eel_river"

gdf = point_to_box_gdf(-124.3066404538428, 40.64402535249519, 512 * 3, 512 * 3)
gdf["filename"] = filename

gdf.to_file(f"/Users/kyledorman/data/estuary/dove/grids/{filename}.geojson", driver="GeoJSON")

# Compute the bounding box of all polygons
minx, miny, maxx, maxy = gdf.total_bounds

# Calculate the center of the bounding box
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Step 5: Create a Folium Map and Overlay
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in gdf.iterrows():
    folium.GeoJson(
        row.geometry,
        name=row.filename,
        # tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Region:"]),
        popup=folium.Popup(row.filename, parse_html=True),
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.5,
        },
    ).add_to(m)

# Display the map (if running in a Jupyter Notebook)
m